In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.datasets.loader import (
    DATASET_IDS,
    load_dataset,
    load_documents,
    load_queries,
    load_qrels,
)

In [3]:
DATASET_IDS

{'main': 'medline/2004/trec-genomics-2005'}

In [4]:
dataset = load_dataset("main")
print(dataset)

Dataset(id='medline/2004/trec-genomics-2005', provides=['docs', 'queries', 'qrels'])


In [5]:
print("Has docs:", dataset.has_docs())
print("Has queries:", dataset.has_queries())
print("Has qrels:", dataset.has_qrels())

Has docs: True
Has queries: True
Has qrels: True


In [6]:
queries = load_queries("main")
qrels = load_qrels("main")

display(queries.head())
display(qrels.head())

print("Queries:", len(queries))
print("Qrels:", len(qrels))

,query_id,text
0,100,"Describe the procedure or methods for how to ""..."
1,101,Describe the procedure or methods for exact re...
2,102,Describe the procedure or methods for differen...
3,103,Describe the procedure or methods for green fl...
4,104,Describe the procedure or methods for how to d...


,query_id,doc_id,relevance,iteration
0,100,10023709,0,0
1,100,10051592,0,0
2,100,10066453,2,0
3,100,10071611,0,0
4,100,10081001,1,0


Queries: 50
Qrels: 39958


In [7]:
docs_sample = load_documents("main", limit=10_000)

display(docs_sample.head())
print(docs_sample.columns.tolist())
print("Sample documents:", len(docs_sample))

[INFO] If you have a local copy of https://dmice.ohsu.edu/trec-gen/data/2004/XML/2004_TREC_XML_MEDLINE_A.gz, you can symlink it here to avoid downloading it again: C:\Users\PC\.ir_datasets\downloads\7858e5b908c25b88e30965b770e9780f
[INFO] [starting] https://dmice.ohsu.edu/trec-gen/data/2004/XML/2004_TREC_XML_MEDLINE_A.gz
[INFO] [finished] https://dmice.ohsu.edu/trec-gen/data/2004/XML/2004_TREC_XML_MEDLINE_A.gz: [15:26] [579MB] [625kB/s]


,doc_id,title,abstract,contents
0,10605436,Concerning the localization of steroids in cen...,"Specific steroid antibodies, by the immunofluo...",Concerning the localization of steroids in cen...
1,10605437,Structural conformation of ciliary dynein arms...,The sliding tubule model of ciliary motion req...,Structural conformation of ciliary dynein arms...
2,10605438,Ouabain binding to renal tubules of the rabbit.,"It is well known that ouabain, a specific inhi...",Ouabain binding to renal tubules of the rabbit...
3,10605439,Isolation and characterization of kinetoplast ...,"We have used restriction endonucleases PstI, E...",Isolation and characterization of kinetoplast ...
4,10605440,Movement of sea urchin sperm flagella.,The motion of the sea urchin sperm flagellum w...,Movement of sea urchin sperm flagella. The mot...


['doc_id', 'title', 'abstract', 'contents']
Sample documents: 10000


In [8]:
print(dataset.docs_count())
print(dataset.has_docs())

3672808
True


In [8]:
print("Docs count:", dataset.docs_count())
print("Queries count:", dataset.queries_count())
print("Qrels count:", dataset.qrels_count())

Docs count: 3672808
Queries count: 50
Qrels count: 39958


In [9]:
print("Docs class:", dataset.docs_cls())
print("Queries class:", dataset.queries_cls())
print("Qrels class:", dataset.qrels_cls())

Docs class: <class 'ir_datasets.datasets.medline.MedlineDoc'>
Queries class: <class 'ir_datasets.formats.base.GenericQuery'>
Qrels class: <class 'ir_datasets.formats.trec.TrecQrel'>


In [10]:
queries = load_queries("main")
qrels = load_qrels("main")

display(queries.head())
display(qrels.head())

print("Queries:", len(queries))
print("Qrels:", len(qrels))

,query_id,text
0,100,"Describe the procedure or methods for how to ""..."
1,101,Describe the procedure or methods for exact re...
2,102,Describe the procedure or methods for differen...
3,103,Describe the procedure or methods for green fl...
4,104,Describe the procedure or methods for how to d...


,query_id,doc_id,relevance,iteration
0,100,10023709,0,0
1,100,10051592,0,0
2,100,10066453,2,0
3,100,10071611,0,0
4,100,10081001,1,0


Queries: 50
Qrels: 39958


In [ ]:
missing_report = docs_sample[["doc_id", "title", "abstract", "contents"]].isna().sum()
display(missing_report)

In [ ]:
empty_contents = docs_sample["contents"].eq("").sum()
print("Empty contents:", empty_contents)

In [ ]:
duplicate_doc_ids = docs_sample["doc_id"].duplicated().sum()
duplicate_query_ids = queries["query_id"].duplicated().sum()

print("Duplicate doc_ids in sample:", duplicate_doc_ids)
print("Duplicate query_ids:", duplicate_query_ids)

In [11]:
relevance_distribution = (
    qrels["relevance"]
    .value_counts()
    .sort_index()
    .rename_axis("relevance")
    .reset_index(name="count")
)

display(relevance_distribution)

,relevance,count
0,0,35374
1,1,2059
2,2,2525


In [15]:
import pandas as pd
from pathlib import Path

audit_df = pd.DataFrame([
    {
        "dataset_name": "Medline TREC Genomics 2005",
        "ir_datasets_id": DATASET_IDS["main"],
        "documents_count_official": dataset.docs_count(),
        "queries_count": len(queries),
        "qrels_count": len(qrels),
        "has_docs": dataset.has_docs(),
        "has_queries": dataset.has_queries(),
        "has_qrels": dataset.has_qrels(),
        "document_fields": str(dataset.docs_cls()),
        "query_fields": str(dataset.queries_cls()),
        "qrels_fields": str(dataset.qrels_cls()),
        "min_relevance": int(qrels["relevance"].min()),
        "max_relevance": int(qrels["relevance"].max()),
        "document_sample_status": "not_loaded_yet_due_to_large_download",
    }
])

display(audit_df)

output_dir = PROJECT_ROOT / "reports"
output_dir.mkdir(parents=True, exist_ok=True)

audit_df.to_csv(output_dir / "dataset_audit.csv", index=False)
relevance_distribution.to_csv(output_dir / "relevance_distribution.csv", index=False)

print("Saved audit reports.")

,dataset_name,ir_datasets_id,documents_count_official,queries_count,qrels_count,has_docs,has_queries,has_qrels,document_fields,query_fields,qrels_fields,min_relevance,max_relevance,document_sample_status
0,Medline TREC Genomics 2005,medline/2004/trec-genomics-2005,3672808,50,39958,True,True,True,<class 'ir_datasets.datasets.medline.MedlineDoc'>,<class 'ir_datasets.formats.base.GenericQuery'>,<class 'ir_datasets.formats.trec.TrecQrel'>,0,2,not_loaded_yet_due_to_large_download


Saved audit reports.


In [ ]:
doc_lengths = docs_sample["contents"].str.split().str.len()

print("Average words:", round(doc_lengths.mean(), 2))
print("Median words:", round(doc_lengths.median(), 2))
print("Max words:", doc_lengths.max())
print("Min words:", doc_lengths.min())

In [ ]:
import pandas as pd

audit_df = pd.DataFrame([
    {
        "dataset_name": "Medline TREC Genomics 2005",
        "ir_datasets_id": DATASET_IDS["main"],
        "documents_count_official": 3672808,
        "queries_count": len(queries),
        "qrels_count": len(qrels),
        "sample_size": len(docs_sample),
        "sample_duplicate_doc_ids": int(duplicate_doc_ids),
        "duplicate_query_ids": int(duplicate_query_ids),
        "sample_empty_documents": int(empty_contents),
        "sample_average_document_words": round(float(doc_lengths.mean()), 2),
        "sample_median_document_words": round(float(doc_lengths.median()), 2),
        "min_relevance": int(qrels["relevance"].min()),
        "max_relevance": int(qrels["relevance"].max()),
    }
])

display(audit_df)

In [13]:
output_dir = PROJECT_ROOT / "reports"
output_dir.mkdir(parents=True, exist_ok=True)

audit_df.to_csv(output_dir / "dataset_audit.csv", index=False)
relevance_distribution.to_csv(output_dir / "relevance_distribution.csv", index=False)

print("Saved audit reports.")

Saved audit reports.


In [16]:
print("Docs count:", dataset.docs_count())
print("Queries count:", dataset.queries_count())
print("Qrels count:", dataset.qrels_count())

Docs count: 3672808
Queries count: 50
Qrels count: 39958


In [17]:
print("Docs class:", dataset.docs_cls())
print("Queries class:", dataset.queries_cls())
print("Qrels class:", dataset.qrels_cls())

Docs class: <class 'ir_datasets.datasets.medline.MedlineDoc'>
Queries class: <class 'ir_datasets.formats.base.GenericQuery'>
Qrels class: <class 'ir_datasets.formats.trec.TrecQrel'>


In [19]:
queries = load_queries("main")
qrels = load_qrels("main")

display(queries.head())
display(qrels.head())

print("Queries:", len(queries))
print("Qrels:", len(qrels))

,query_id,text
0,100,"Describe the procedure or methods for how to ""..."
1,101,Describe the procedure or methods for exact re...
2,102,Describe the procedure or methods for differen...
3,103,Describe the procedure or methods for green fl...
4,104,Describe the procedure or methods for how to d...


,query_id,doc_id,relevance,iteration
0,100,10023709,0,0
1,100,10051592,0,0
2,100,10066453,2,0
3,100,10071611,0,0
4,100,10081001,1,0


Queries: 50
Qrels: 39958


In [20]:
relevance_distribution = (
    qrels["relevance"]
    .value_counts()
    .sort_index()
    .rename_axis("relevance")
    .reset_index(name="count")
)

display(relevance_distribution)

,relevance,count
0,0,35374
1,1,2059
2,2,2525


In [25]:
import pandas as pd
from pathlib import Path

audit_df = pd.DataFrame([
    {
        "dataset_name": "Medline TREC Genomics 2005",
        "ir_datasets_id": DATASET_IDS["main"],
        "documents_count_official": dataset.docs_count(),
        "queries_count": len(queries),
        "qrels_count": len(qrels),
        "has_docs": dataset.has_docs(),
        "has_queries": dataset.has_queries(),
        "has_qrels": dataset.has_qrels(),
        # "document_fields": str(dataset.docs_cls()),
        # "query_fields": str(dataset.queries_cls()),
        # "qrels_fields": str(dataset.qrels_cls()),
        "document_schema": ["doc_id", "title", "abstract"],
        "query_schema": ["query_id", "text"],
        "qrels_schema": ["query_id", "doc_id", "relevance", "iteration"],
        "min_relevance": int(qrels["relevance"].min()),
        "max_relevance": int(qrels["relevance"].max()),
        "document_sample_status": "not_loaded_yet_due_to_large_download",
    }
])

display(audit_df)

output_dir = PROJECT_ROOT / "reports"
output_dir.mkdir(parents=True, exist_ok=True)

audit_df.to_csv(output_dir / "dataset_audit.csv", index=False)
relevance_distribution.to_csv(output_dir / "relevance_distribution.csv", index=False)

print("Saved audit reports.")

,dataset_name,ir_datasets_id,documents_count_official,queries_count,qrels_count,has_docs,has_queries,has_qrels,document_schema,query_schema,qrels_schema,min_relevance,max_relevance,document_sample_status
0,Medline TREC Genomics 2005,medline/2004/trec-genomics-2005,3672808,50,39958,True,True,True,"[doc_id, title, abstract]","[query_id, text]","[query_id, doc_id, relevance, iteration]",0,2,not_loaded_yet_due_to_large_download


Saved audit reports.


In [1]:
import ir_datasets

dataset = ir_datasets.load("medline/2004/trec-genomics-2005")

docs_iter = dataset.docs_iter()
queries_iter = dataset.queries_iter()
qrels_iter = dataset.qrels_iter()

first_doc = next(docs_iter)
first_query = next(queries_iter)
first_qrel = next(qrels_iter)

print(first_doc)
print(first_query)
print(first_qrel)

MedlineDoc(doc_id='10605436', title='Concerning the localization of steroids in centrioles and basal bodies by immunofluorescence.', abstract='Specific steroid antibodies, by the immunofluorescence technique, regularly reveal fluorescent centrioles and cilia-bearing basal bodies in target and nontarget cells. Although the precise identity of the immunoreactive steroid substance has not yet been established, it seems noteworthy that exogenous steroids can be vitally concentrated by centrioles, perhaps by exchange with steroids already present at this level. This unexpected localization suggests that steroids may affect cell growth and differentiation in some way different from the two-step receptor mechanism.')
GenericQuery(query_id='100', text='Describe the procedure or methods for how to "open up" a cell through a process called "electroporation."')
TrecQrel(query_id='100', doc_id='10023709', relevance=0, iteration='0')
